In [ ]:
# Library imports
from dotenv import load_dotenv
import os
import openai
import pandas as pd
import json
import time

In [ ]:
# API Key
load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")

In [ ]:
# File paths
BASE_DIR    = r"F:\\workspace1"
GOV_PATH    = os.path.join(BASE_DIR, "trait_data_gov.csv")
COM_PATH    = os.path.join(BASE_DIR, "trait_data_com.csv")
UNIV_PATH   = os.path.join(BASE_DIR, "trait_data_univ.csv")
PHASE0_PATH = os.path.join(BASE_DIR, "formatted_answer_sheet_phase0.csv")

In [ ]:
# New entrant profiles
NEW_ENTRANTS = {
    147: {
        "type": "com",
        "data": {
            "ID": 147,
            "Name": "Cryovexalit Launch",
            "Main Industry": "Launch Vehicle Manufacturers, Launch Vehicle Equipment and Software",
            "Sub Industry": "Small launch vehicles, payload deployment systems, launch operations",
            "Year": 2020,
            "Employees": "501~1000",
            "Company Type": "Startup",
            "Revenue": "$50M ~ $200M",
            "Operating Income": "N/A",
            "Listed": "Unlisted",
            "Enterprise Value": "approx. $150M",
            "Funding": "Series C, Total Funding: $500M",
            "Company Info": "Early-stage launch vehicle startup developing dedicated small satellite launch services with a focus on rapid launch cadence and low-cost access to LEO. Currently in advanced vehicle testing phase with first commercial launch planned.",
            "Products Services": "Small-lift launch vehicle (payload capacity: 300kg to LEO), dedicated rideshare services, payload integration support",
            "Core Technology": "LOX/methane propulsion, lightweight carbon composite airframe, autonomous flight termination system, rapid launch pad setup",
            "Patents": 15,
            "Location": "Mojave, California, United States"
        }
    },
    148: {
        "type": "com",
        "data": {
            "ID": 148,
            "Name": "Aeriqonax Systems",
            "Main Industry": "Launch Vehicle Equipment and Software, Launch Support Services and Equipment, Launch Vehicle Design and Manufacturing Support",
            "Sub Industry": "Avionics, guidance and navigation systems, launch vehicle integration support",
            "Year": 2008,
            "Employees": "501~1000",
            "Company Type": "SME",
            "Revenue": "$100M ~ $500M",
            "Operating Income": "$10M ~ $50M",
            "Listed": "Unlisted",
            "Enterprise Value": "approx. $400M",
            "Funding": "Bootstrapped with SBIR grants",
            "Company Info": "Mid-sized supplier of avionics and guidance systems for launch vehicles with over 15 years of experience supporting both government and commercial launch programs. Has established subcontracting relationships with prime contractors and holds relevant government certifications.",
            "Products Services": "Flight avionics units, inertial navigation systems, launch vehicle integration and test support, ground support equipment",
            "Core Technology": "Radiation-hardened avionics, inertial measurement units (IMU), real-time flight software, FAR-compliant manufacturing processes",
            "Patents": 20,
            "Location": "Huntsville, Alabama, United States"
        }
    },
    149: {
        "type": "univ",
        "data": {
            "ID": 149,
            "Name": "TerraOrbit University (TOU)",
            "Category": "Public",
            "Year": 1965,
            "R&D Expenditure": "$1,500,000",
            "Professors": 45,
            "Graduate Students": 320,
            "Laboratory": "Propulsion Research Laboratory, Launch Vehicle Structures Lab, Guidance and Control Systems Lab, Small Satellite Design Lab",
            "Research Area": "Launch vehicle propulsion, structural dynamics, guidance navigation and control, small satellite systems, composite materials for aerospace",
            "Core Strength": "Focused aerospace engineering program with strong industry linkages in the Midwest aerospace corridor. Emphasis on applied research and technology transfer relevant to launch vehicle subsystems. Active SBIR collaboration history with regional aerospace firms and government field centers.",
            "Papers": 3500,
            "Patents": 12,
            "Location": "Columbus, Ohio, United States"
        }
    }
}

NEW_ENTRANT_IDS   = list(NEW_ENTRANTS.keys())
NEW_ENTRANT_NAMES = ", ".join(
    f"{v['data']['Name']} (ID: {k})" for k, v in NEW_ENTRANTS.items()
)

In [ ]:
# Policy definitions
mission_oriented_policy = {
    "budget_structure": "Consists of large-scale contracts with significant funding allocations.",
    "schedule": "Long-term schedules are assigned with no allowance for mid-course failure.",
    "development_cost": {
        "funding": "Fully funded by the government.",
        "payment_structure": "Government pays primary development costs.",
        "contract_type": "Cost-plus contracts, with the government covering any cost overruns."
    },
    "mass_production": "Production is government-owned, operated, maintained, and utilized.",
    "contract_form": {
        "type": "Traditional government contracts under the Federal Acquisition Regulation (FAR).",
        "development_contracts": "Major development and production contracts are issued as cost-plus contracts."
    },
    "role_distribution": {
        "lead": "Government-led development with total management responsibility.",
        "industry_role": "Private companies act as subcontractors manufacturing parts and systems.",
        "specs": "Government defines all technical specifications and design requirements."
    },
    "ownership_and_operations": {
        "assets": "Developed rockets and spacecraft are owned by the government.",
        "mission_operations": "Conducted under government supervision with limited commercial use."
    },
    "program_management": {
        "approach": "Strict supervision and process-centered management.",
        "collaboration": "Close collaboration between the government and contractors.",
        "review_approval": "Government review and approval required at every stage."
    },
    "design_approach": "Designs are tailored exactly to requirements, reflecting government and agency standards.",
    "development_risk": "Solely borne by the government, including cost overruns and schedule delays.",
    "operational_risk": "Government is responsible for losses and crew casualties in the event of mission failure.",
    "commercial_risk": {
        "market_relation": "Independent of market conditions.",
        "compensation": "Government provides compensation.",
        "market_use": "Launch vehicles are government-exclusive systems, unrelated to the commercial market.",
        "profit": "Contractors receive predetermined profit from the government."
    },
    "intellectual_property": {
        "accumulation": "Technology and know-how accumulated within government and its contractors.",
        "ownership": "Patents and design information are controlled by the government.",
        "transfer": "Only selected technologies are transferred to the private sector."
    },
    "technology_transfer": {
        "method": "Advanced technologies developed by the government are disseminated through spinoff programs.",
        "private_sector_role": "Private entities apply technologies post-development rather than directly participating in R&D."
    }
}

diffusion_oriented_policy = {
    "budget_structure": "Composed of multiple small-scale contracts with limited funding per unit.",
    "schedule": "Short-term schedules are set, and failure to meet key milestones results in contract termination.",
    "development_cost": {
        "funding": "Cost-sharing between government and private sector.",
        "payment_structure": "Government provides fixed funding; contractors must invest their own capital.",
        "contract_type": "Performance-based contracts, payment issued only upon milestone achievement."
    },
    "mass_production": "Ownership, operation, and maintenance lie with private companies. Government purchases services as needed.",
    "contract_form": {
        "type": "Agreements under space law, allowing flexibility.",
        "development_contracts": "Issued as fixed-price contracts to private entities."
    },
    "role_distribution": {
        "lead": "Joint development between government and private companies.",
        "government_role": "Sets high-level goals and funding; provides technical advice and evaluation only.",
        "industry_role": "Private firms lead design and development efforts."
    },
    "ownership_and_operations": {
        "assets": "Rockets and spacecraft are owned by private companies.",
        "mission_operations": "Government uses them as a service; private firms may serve commercial clients as well."
    },
    "program_management": {
        "approach": "Light supervision with performance-based oversight.",
        "government_involvement": "Validates major milestone achievements only.",
        "decision_making": "Detailed design and development decisions are at company discretion."
    },
    "design_approach": "Designs are goal-oriented and flexible, with multiple approaches allowed. Government specifies outcomes, not methods.",
    "development_risk": {
        "burden": "Shared between government and private companies.",
        "failure_consequence": "Government withholds remaining support; remaining losses are borne by the company."
    },
    "operational_risk": {
        "responsibility": "Private sector under commercial launch oversight agencies.",
        "failure_consequence": "Contractors bear full responsibility for re-launch costs in case of mission failure."
    },
    "commercial_risk": {
        "burden": "Fully borne by the private sector.",
        "ownership": "Launch vehicles are privately owned.",
        "market_responsibility": "Private companies are responsible for securing commercial demand.",
        "revenue_opportunity": "Firms may earn additional revenue from launching third-party satellites."
    },
    "intellectual_property": {
        "ownership": "Technology and know-how are accumulated within private firms.",
        "government_access": "Government retains limited access to necessary data only.",
        "control": "Intellectual property rights remain with the private companies."
    },
    "technology_transfer": {
        "development": "Private firms develop technologies to meet government needs.",
        "government_support": "Government may provide test facilities and technical expertise when necessary.",
        "utilization": "Firms retain and commercialize the technologies after development."
    }
}

SCENARIO_CONFIG = {
    "Mission": {
        "policy":             mission_oriented_policy,
        "policy_name":        "Mission-oriented",
        "phase0_col_partner": "M_Partner",
        "phase0_col_reason":  "M_Partner_Reason"
    },
    "Diffusion": {
        "policy":             diffusion_oriented_policy,
        "policy_name":        "Diffusion-oriented",
        "phase0_col_partner": "D_Partner",
        "phase0_col_reason":  "D_Partner_Reason"
    }
}

In [ ]:
# Data loading
def load_trait_data():
    gov_data  = pd.read_csv(GOV_PATH,  encoding='utf-8-sig')
    com_data  = pd.read_csv(COM_PATH,  encoding='utf-8-sig')
    univ_data = pd.read_csv(UNIV_PATH, encoding='utf-8-sig')
    print(f"Government: {len(gov_data)} | Companies: {len(com_data)} | Universities: {len(univ_data)}")
    return gov_data, com_data, univ_data

def load_phase0():
    df = pd.read_csv(PHASE0_PATH, encoding='utf-8-sig')
    print(f"Phase 0 loaded: {len(df)} rows")
    return df

In [ ]:
# System message
def create_system_message(row, data_type):
    if data_type == "gov":
        return (
            f"You are a decision-maker for government agency ID {row['ID']}, {row['Name']}. "
            f"Your government agency has the following characteristics: "
            f"Category is {row['Category']}, established in {row['Year']}, "
            f"has {row['Employees']} employees, yearly budget is {row['Budget']}, "
            f"organization information is {row['Organization Info']}, "
            f"main mission is {row['Mission']}, "
            f"subordinate centers are {row['Subordinate Centers']}, "
            f"total number of papers published is {row['Papers']}, "
            f"number of space technology-related patents (B64G) is {row['Patents']}, "
            f"and headquarters location is {row['Location']}."
        )
    elif data_type == "com":
        return (
            f"You are a decision-maker for company ID {row['ID']}, {row['Name']}. "
            f"Your company has the following characteristics: "
            f"Main industry is {row['Main Industry']}, sub industry is {row['Sub Industry']}, "
            f"established in {row['Year']}, has {row['Employees']} employees, "
            f"company type is {row['Company Type']}, revenue is {row['Revenue']}, "
            f"operating income is {row['Operating Income']}, listing status is {row['Listed']}, "
            f"enterprise value is {row['Enterprise Value']}, funding level is {row['Funding']}, "
            f"company information is {row['Company Info']}, "
            f"main products and services are {row['Products Services']}, "
            f"core technology is {row['Core Technology']}, "
            f"number of space technology-related patents (B64G) is {row['Patents']}, "
            f"and headquarters location is {row['Location']}."
        )
    elif data_type == "univ":
        return (
            f"You are a decision-maker for university ID {row['ID']}, {row['Name']}. "
            f"Your university has the following characteristics: "
            f"Category (private/public) is {row['Category']}, established in {row['Year']}, "
            f"annual R&D expenditure is {row['R&D Expenditure']}, "
            f"total number of aerospace department professors is {row['Professors']}, "
            f"number of aerospace graduate students is {row['Graduate Students']}, "
            f"representative laboratory is {row['Laboratory']}, "
            f"main research area is {row['Research Area']}, "
            f"core strength of the aerospace department is {row['Core Strength']}, "
            f"total number of papers published is {row['Papers']}, "
            f"number of space technology-related patents (B64G) is {row['Patents']}, "
            f"and university location is {row['Location']}."
        )

In [ ]:
# Partner info with new entrants added
def create_partner_info(gov_data, com_data, univ_data):
    info = "The following is information about potential collaboration partners:\n\n"

    info += "=== Government Agencies ===\n"
    for _, row in gov_data.iterrows():
        info += (
            f"ID {row['ID']}: {row['Name']} - Category: {row['Category']}, "
            f"Established: {row['Year']}, Employees: {row['Employees']}, "
            f"Yearly budget: {row['Budget']}, Organization Info: {row['Organization Info']}, "
            f"Mission: {row['Mission']}, Subordinate Centers: {row['Subordinate Centers']}, "
            f"Papers: {row['Papers']}, Patents: {row['Patents']}, Location: {row['Location']}\n"
        )

    info += "\n=== Companies ===\n"
    for _, row in com_data.iterrows():
        info += (
            f"ID {row['ID']}: {row['Name']} - Industry: {row['Main Industry']}/{row['Sub Industry']}, "
            f"Established: {row['Year']}, Employees: {row['Employees']}, "
            f"Type: {row['Company Type']}, Revenue: {row['Revenue']}, "
            f"Operating Income: {row['Operating Income']}, Listed: {row['Listed']}, "
            f"Enterprise Value: {row['Enterprise Value']}, Funding: {row['Funding']}, "
            f"Company Info: {row['Company Info']}, "
            f"Main Product & Service: {row['Products Services']}, "
            f"Core Technology: {row['Core Technology']}, "
            f"Patents: {row['Patents']}, Location: {row['Location']}\n"
        )
    for agent_id, e in NEW_ENTRANTS.items():
        if e["type"] == "com":
            d = e["data"]
            info += (
                f"ID {d['ID']}: {d['Name']} [NEW ENTRANT] - "
                f"Industry: {d['Main Industry']}/{d['Sub Industry']}, "
                f"Established: {d['Year']}, Employees: {d['Employees']}, "
                f"Type: {d['Company Type']}, Revenue: {d['Revenue']}, "
                f"Operating Income: {d['Operating Income']}, Listed: {d['Listed']}, "
                f"Enterprise Value: {d['Enterprise Value']}, Funding: {d['Funding']}, "
                f"Company Info: {d['Company Info']}, "
                f"Main Product & Service: {d['Products Services']}, "
                f"Core Technology: {d['Core Technology']}, "
                f"Patents: {d['Patents']}, Location: {d['Location']}\n"
            )

    info += "\n=== Universities ===\n"
    for _, row in univ_data.iterrows():
        info += (
            f"ID {row['ID']}: {row['Name']} - Category(Public/Private): {row['Category']}, "
            f"Established: {row['Year']}, R&D Expenditure: {row['R&D Expenditure']}, "
            f"Professors: {row['Professors']}, Graduate Students: {row['Graduate Students']}, "
            f"Main Laboratory: {row['Laboratory']}, Research Area: {row['Research Area']}, "
            f"Core Strength: {row['Core Strength']}, "
            f"Papers: {row['Papers']}, Patents: {row['Patents']}, Location: {row['Location']}\n"
        )
    for agent_id, e in NEW_ENTRANTS.items():
        if e["type"] == "univ":
            d = e["data"]
            info += (
                f"ID {d['ID']}: {d['Name']} [NEW ENTRANT] - "
                f"Category(Public/Private): {d['Category']}, "
                f"Established: {d['Year']}, R&D Expenditure: {d['R&D Expenditure']}, "
                f"Professors: {d['Professors']}, Graduate Students: {d['Graduate Students']}, "
                f"Main Laboratory: {d['Laboratory']}, Research Area: {d['Research Area']}, "
                f"Core Strength: {d['Core Strength']}, "
                f"Papers: {d['Papers']}, Patents: {d['Patents']}, Location: {d['Location']}\n"
            )
    return info

In [ ]:
# Memory builder
def build_memory(agent_id, prev_result_df, partner_col, reason_col):
    if agent_id in NEW_ENTRANT_IDS:
        return "[New Entrant] This is your first round of collaboration selection in this network."
    row = prev_result_df[prev_result_df['ID'] == agent_id]
    if row.empty:
        return "No prior collaboration record available."
    partners = str(row.iloc[0][partner_col]).strip()
    reasons  = str(row.iloc[0][reason_col]).strip()
    return (
        f"[Prior Round Memory]\n"
        f"In the previous round, your organization selected the following collaboration partners:\n"
        f"Partner IDs: {partners}\n"
        f"Reasons for selection:\n{reasons}\n"
    )

In [ ]:
# Prompt builder
def create_user_prompt_entrant(policy_dict, policy_name,
                                partner_info, memory_str,
                                phase, is_new_entrant=False):
    policy_str = json.dumps(policy_dict, indent=2, ensure_ascii=False)

    if phase == 1:
        if is_new_entrant:
            shock_context = (
                "[New Entrant]\n"
                "Your organization has just entered the U.S. space launch vehicle collaboration network. "
                "Please select your initial collaboration partners based on your organizational profile "
                "and the current policy environment.\n"
            )
        else:
            shock_context = (
                f"[New Entrant Arrival Shock]\n"
                f"Three new organizations have entered the collaboration network: {NEW_ENTRANT_NAMES}. "
                f"Their profiles are included in the partner list below. "
                f"Please reconsider your collaboration strategy accordingly.\n"
            )
    elif phase == 2:
        shock_context = (
            f"[Adaptation Round]\n"
            f"The three new entrants ({NEW_ENTRANT_NAMES}) remain active in the network. "
            f"Please further adjust your collaboration strategy based on the evolving network conditions.\n"
        )
    else:
        shock_context = (
            f"[Stabilization Round]\n"
            f"The three new entrants ({NEW_ENTRANT_NAMES}) remain active in the network. "
            f"Please finalize your collaboration strategy under the stabilized network conditions.\n"
        )

    prompt = f"""
The government will be operating research and development and will be forming markets in the following areas:
Launch Services, Launch Site Operators, Launch Support Services and Equipment, Launch Vehicle Manufacturers,
Launch Vehicle Equipment and Software, Manufacturers of Spacecraft Deployment Systems,
Launch Vehicle Test Equipment, Launch Vehicle Design and Manufacturing Support.

{memory_str}

{shock_context}

Current policy direction ({policy_name}): {policy_str}

The profiles of potential collaboration partners (including new entrants):
{partner_info}

Based on your prior collaboration history, the new entrant arrival, and the profiles of potential partners,
please decide your collaboration partners by:
1. Partners to MAINTAIN from the previous round (with reasons) — N/A if new entrant
2. Partners to DISCONTINUE from the previous round (with reasons) — N/A if new entrant
3. NEW partners to add, including any of the new entrants if appropriate (with reasons)

Please respond using the following format:
- Selected Partner ID(s): (all final partners, numbers only, comma-separated)
- Maintained Partner IDs:
- Discontinued Partner IDs:
- New Partner IDs:
- Reason(s) for Each Decision:
- Overall Considerations for Your Decision:
"""
    return prompt

In [ ]:
# OpenAI call
def ask_openai(system_message, user_message, max_retries=3):
    for attempt in range(max_retries):
        try:
            response = openai.ChatCompletion.create(
                model="gpt-4.1",
                messages=[
                    {"role": "system", "content": system_message},
                    {"role": "user",   "content": user_message},
                ],
                temperature=1.0,
                max_tokens=9000
            )
            return response.choices[0].message["content"].strip()
        except Exception as e:
            print(f"API call failed (attempt {attempt+1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                time.sleep(2)
            else:
                return "API_ERROR"

In [ ]:
# Result parser
def parse_partner_ids(response_text):
    for line in response_text.split('\n'):
        if 'Selected Partner ID' in line:
            parts = line.split(':', 1)
            if len(parts) > 1:
                ids_str = parts[1].strip()
                ids_str = ''.join(c for c in ids_str if c.isdigit() or c in ', ')
                return ids_str.strip()
    return ""

In [ ]:
# Phase runner
def run_phase(phase, scenario_key, gov_data, com_data, univ_data,
              prev_result_df, partner_info):
    cfg = SCENARIO_CONFIG[scenario_key]

    if phase == 1:
        mem_partner_col = cfg["phase0_col_partner"]
        mem_reason_col  = cfg["phase0_col_reason"]
    else:
        mem_partner_col = "Partner_IDs"
        mem_reason_col  = "Full_Response"

    all_data = []
    for _, row in gov_data.iterrows():
        all_data.append((row.to_dict(), "gov", False))
    for _, row in com_data.iterrows():
        all_data.append((row.to_dict(), "com", False))
    for _, row in univ_data.iterrows():
        all_data.append((row.to_dict(), "univ", False))
    for agent_id, e in NEW_ENTRANTS.items():
        all_data.append((e["data"], e["type"], True))

    total   = len(all_data)
    results = []

    for i, (row, data_type, is_new) in enumerate(all_data):
        agent_id = row['ID']
        label    = " [NEW ENTRANT]" if is_new else ""
        print(f"[{scenario_key}] Phase {phase} | Processing: ID {agent_id} ({i+1}/{total}){label}")

        system_msg = create_system_message(row, data_type)
        memory_str = build_memory(agent_id, prev_result_df, mem_partner_col, mem_reason_col)
        user_msg   = create_user_prompt_entrant(
            policy_dict    = cfg["policy"],
            policy_name    = cfg["policy_name"],
            partner_info   = partner_info,
            memory_str     = memory_str,
            phase          = phase,
            is_new_entrant = is_new
        )

        response    = ask_openai(system_msg, user_msg)
        partner_ids = parse_partner_ids(response)

        results.append({
            "ID":            agent_id,
            "Partner_IDs":   partner_ids,
            "Full_Response": response
        })
        time.sleep(1)

    result_df = pd.DataFrame(results)
    out_path  = os.path.join(BASE_DIR, f"answer_sheet_S2_{scenario_key}_phase{phase}.csv")
    result_df.to_csv(out_path, index=False, encoding='utf-8-sig')
    print(f"Saved: {out_path}")
    return result_df

In [ ]:
# Main
def main():
    print("=" * 60)
    print("S2. New Entrant Arrival Simulation")
    print(f"New Entrants: {NEW_ENTRANT_NAMES}")
    print("=" * 60)

    gov_data, com_data, univ_data = load_trait_data()
    phase0_df    = load_phase0()
    partner_info = create_partner_info(gov_data, com_data, univ_data)

    for scenario_key in ["Mission", "Diffusion"]:
        print(f"\n{'='*60}")
        print(f"Scenario: {scenario_key}")
        print(f"{'='*60}")

        prev_df = phase0_df

        for phase in [1, 2, 3]:
            print(f"\n--- Phase {phase} Start ---")
            prev_df = run_phase(
                phase          = phase,
                scenario_key   = scenario_key,
                gov_data       = gov_data,
                com_data       = com_data,
                univ_data      = univ_data,
                prev_result_df = prev_df,
                partner_info   = partner_info
            )
            print(f"--- Phase {phase} Complete ---")

    print("\nAll simulations complete.")

In [ ]:
# Execute
main()